# Agent에 메모리(Memory) 추가

LangGraph에서 에이전트(Agent)는 도구를 사용하여 사용자 질문에 답변할 수 있지만, 기본적으로 이전 대화의 맥락(context)을 기억하지 못합니다. 이는 멀티턴(multi-turn) 대화를 진행하는 데 큰 제약이 됩니다. 예를 들어, 사용자가 자신의 이름을 알려준 뒤 다시 물어보면 에이전트는 이를 기억하지 못합니다.

LangGraph는 **persistent checkpointing**을 통해 이 문제를 해결합니다. 그래프를 컴파일할 때 `checkpointer`를 제공하고 그래프를 호출할 때 `thread_id`를 제공하면, LangGraph는 각 단계 후 **상태를 자동으로 저장**합니다. 동일한 `thread_id`를 사용하여 그래프를 다시 호출하면, 저장된 상태를 로드하여 에이전트가 이전에 중단한 지점에서 대화를 이어갈 수 있습니다.

이번 튜토리얼에서는 pre-built 되어 있는 `ToolNode`와 `tools_condition`을 활용하여 도구 사용이 가능한 에이전트에 메모리를 추가합니다. checkpointing은 LangChain의 단순한 메모리 기능보다 훨씬 강력하며, 이 튜토리얼을 완수하면 자연스럽게 그 차이를 확인할 수 있습니다.

> 참고 문서:
> - [ToolNode](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.tool_node.ToolNode): 도구 호출을 위한 노드
> - [tools_condition](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.tool_node.tools_condition): 도구 호출 여부에 따른 조건 분기
> - [LangGraph Persistence](https://langchain-ai.github.io/langgraph/concepts/persistence/): 체크포인팅 개념 문서

## 환경 설정

LangGraph 튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, `langchain_teddynote`의 로깅 기능을 활성화하여 LangSmith에서 실행 추적을 확인할 수 있도록 합니다.

아래 코드는 환경 변수를 로드하고 LangSmith 프로젝트를 설정합니다.

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)
# 추적을 위한 프로젝트 이름 설정
logging.langsmith("LangGraph-V1-Tutorial")

LangChain/LangSmith API Key가 설정되지 않았습니다. 참고: https://wikidocs.net/250954


---

## MemorySaver 체크포인터 생성

멀티턴(multi-turn) 대화를 가능하게 하기 위해 **checkpointing**을 추가합니다. 체크포인터는 그래프의 각 단계에서 상태를 저장하여, 이후 동일한 대화를 이어서 진행할 수 있게 합니다. `MemorySaver`는 인메모리 체크포인터로, 메모리에 상태를 저장하므로 개발 및 테스트 환경에서 사용하기 적합합니다.

프로덕션 환경에서는 서버 재시작 시에도 상태가 유지되어야 하므로 `PostgresSaver`나 `SqliteSaver` 같은 영구 저장소 기반 체크포인터를 사용하는 것이 좋습니다.

> 참고 문서: [LangGraph Persistence](https://langchain-ai.github.io/langgraph/concepts/persistence/)

아래 코드에서는 `MemorySaver` 체크포인터를 생성합니다.

In [2]:
from langgraph.checkpoint.memory import MemorySaver

# 메모리 저장소 생성
memory = MemorySaver()

---

## 에이전트 그래프 구성

이제 검색 도구를 사용하는 에이전트 그래프를 구성합니다. 그래프는 **State 정의**, **도구 및 LLM 바인딩**, **노드 추가**, **엣지 연결**의 4단계로 구성됩니다. `State`에는 `add_messages` 리듀서가 적용된 `messages` 필드를 정의하여, 새 메시지가 기존 리스트에 자동으로 추가되도록 합니다.

`ToolNode`는 도구 호출을 처리하는 pre-built 노드이며, `tools_condition`은 LLM 응답에 도구 호출이 포함되었는지에 따라 분기를 결정하는 조건 함수입니다. 이 두 컴포넌트를 사용하면 도구 호출 로직을 직접 구현할 필요 없이 간단하게 에이전트를 구축할 수 있습니다.

아래 코드에서는 상태를 정의하고, 검색 도구와 LLM을 바인딩한 뒤, 그래프의 노드와 엣지를 구성합니다.

In [3]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain_teddynote.tools.tavily import TavilySearch
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


########## 1. 상태 정의 ##########
class State(TypedDict):
    """에이전트의 상태를 정의하는 타입

    messages: 대화 메시지 리스트
    - add_messages 리듀서를 통해 새 메시지가 추가됩니다
    """

    messages: Annotated[list, add_messages]


########## 2. 도구 정의 및 바인딩 ##########
# 검색 도구 초기화 (최대 3개 결과)
tool = TavilySearch(max_results=3)
tools = [tool]

# LLM 초기화
# OpenAI 키 사용 시 gpt-5.2, gpt-4.1-mini 등으로 변경 가능
llm = init_chat_model("claude-sonnet-4-5")

# 도구와 LLM 결합
llm_with_tools = llm.bind_tools(tools)


########## 3. 노드 추가 ##########
def chatbot(state: State):
    """챗봇 노드 함수

    현재 상태의 메시지를 LLM에 전달하고 응답을 반환합니다.
    """
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


# 상태 그래프 생성
graph_builder = StateGraph(State)

# 챗봇 노드 추가
graph_builder.add_node("chatbot", chatbot)

# 도구 노드 생성 및 추가
tool_node = ToolNode(tools=[tool])
graph_builder.add_node("tools", tool_node)

# 조건부 엣지: 도구 호출 여부에 따라 분기
graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)

########## 4. 엣지 추가 ##########
# tools -> chatbot: 도구 실행 후 다시 챗봇으로
graph_builder.add_edge("tools", "chatbot")

# START -> chatbot: 시작점에서 챗봇으로
graph_builder.add_edge(START, "chatbot")

---

## 그래프 컴파일 (체크포인터 적용)

그래프를 컴파일할 때 `checkpointer` 파라미터에 생성한 `MemorySaver`를 전달합니다. 이렇게 하면 그래프가 각 노드를 실행할 때마다 자동으로 상태가 체크포인트에 저장됩니다. 체크포인터가 없는 그래프는 각 호출이 독립적이지만, 체크포인터가 있으면 호출 간에 상태가 유지됩니다.

동일한 `thread_id`로 그래프를 다시 호출하면, 저장된 상태를 불러와 이전 대화를 이어서 진행할 수 있습니다. 이것이 LangGraph에서 메모리를 구현하는 핵심 메커니즘입니다.

아래 코드에서는 체크포인터를 적용하여 그래프를 컴파일합니다.

In [4]:
# 체크포인터를 적용하여 그래프 컴파일
graph = graph_builder.compile(checkpointer=memory)

---

## 그래프 시각화

그래프의 연결 구조는 이전 `LangGraph-Agent` 튜토리얼과 동일합니다. `chatbot` 노드에서 도구 호출이 필요하면 `tools` 노드로 이동하고, 도구 실행 결과를 받아 다시 `chatbot` 노드로 돌아옵니다. 차이점은 이번에 추가된 체크포인터가 그래프의 각 노드를 처리하면서 `State`를 자동으로 저장한다는 것입니다.

아래 다이어그램은 에이전트 그래프의 구조를 보여줍니다.

In [ ]:
from IPython.display import Image

# 그래프 시각화 (Excalidraw로 생성된 PNG 이미지)
Image(filename="assets/04-agent-memory-graph.png")

---

## 멀티턴 대화 테스트

### RunnableConfig 설정

`RunnableConfig`을 정의하고 `recursion_limit`과 `thread_id`를 설정합니다. `recursion_limit`은 최대 방문할 노드 수를 제한하여 무한 루프를 방지하며, `thread_id`는 대화 세션을 구분하는 데 사용됩니다.

`thread_id`가 같으면 동일한 대화 세션으로 취급되어 이전 대화 내용이 유지됩니다. 반대로, `thread_id`가 다르면 별도의 세션으로 관리됩니다. 이를 통해 여러 사용자 또는 여러 대화를 독립적으로 관리할 수 있습니다.

아래 코드에서는 `thread_id`를 `"1"`로 설정한 Config를 생성합니다.

In [6]:
from langchain_core.runnables import RunnableConfig
from langchain_teddynote.messages import stream_graph

# Config 설정: thread_id로 대화 세션을 구분
config = RunnableConfig(
    recursion_limit=10,  # 최대 10개의 노드까지 방문. 그 이상은 RecursionError 발생
    configurable={"thread_id": "1"},  # 스레드 ID 설정
)

아래 코드에서는 첫 번째 대화에서 이름을 알려주고, 이어지는 대화에서 에이전트가 이전 대화를 기억하는지 확인합니다. `stream_graph()` 함수를 사용하여 각 노드의 실행 결과를 스트리밍 방식으로 출력합니다.

In [7]:
# 첫 번째 질문: 자기소개
inputs = {
    "messages": [
        (
            "user",
            "내 이름은 `테디노트` 입니다. YouTube 채널을 운영하고 있어요. 만나서 반가워요",
        )
    ]
}

# 스트리밍 실행
stream_graph(graph, inputs=inputs, config=config)


🔄 Node: chatbot 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
안녕하세요,

 테디노트님

! 만

나서 반가워요!

🎬

...

야기 

나

눌

 수 있습니다.

😊

In [8]:
# 이어지는 질문: 이전 대화 기억 확인
inputs = {"messages": [("user", "내 이름이 뭐라고 했지?")]}

# 같은 thread_id로 스트리밍 실행
stream_graph(graph, inputs=inputs, config=config)


🔄 Node: chatbot 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
테

디노트라

고 하

셨습

니다! 

...

고

 소

개해 주

셨어

요.

### 다른 thread_id로 테스트

이번에는 `RunnableConfig`의 `thread_id`를 변경한 뒤, 이전 대화 내용을 기억하고 있는지 물어보겠습니다. `thread_id`가 다르면 별도의 대화 세션으로 취급되므로, 이전 대화 내용을 기억하지 못합니다.

이를 통해 `thread_id`가 대화 세션을 구분하는 역할을 한다는 것을 확인할 수 있습니다. 실제 서비스에서는 사용자별로 고유한 `thread_id`를 부여하여 각 사용자의 대화를 독립적으로 관리합니다.

아래 코드에서는 `thread_id`를 `"2"`로 변경하여 새로운 세션에서 이전 대화를 기억하는지 테스트합니다.

In [9]:
# 다른 thread_id로 Config 설정
config_2 = RunnableConfig(
    recursion_limit=10,  # 최대 10개의 노드까지 방문
    configurable={"thread_id": "2"},  # 다른 스레드 ID
)

# 이전 대화를 기억하는지 확인
inputs = {"messages": [("user", "내 이름이 뭐라고 했지?")]}

# 스트리밍 실행 (다른 thread_id이므로 이전 대화를 기억하지 못함)
stream_graph(graph, inputs=inputs, config=config_2)


🔄 Node: chatbot 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
죄송하

지만,

 우

리의

 대

...

중

에 기

억하

고 있

겠습니다!

---

## 스냅샷: 저장된 State 확인

지금까지 두 개의 다른 스레드에서 몇 개의 체크포인트를 만들었습니다. `Checkpoint`에는 현재 상태 값(values), 해당 구성(config), 그리고 처리할 다음 노드(next)가 포함되어 있습니다. 이러한 정보를 활용하면 그래프의 실행 상태를 정확하게 파악하고 디버깅할 수 있습니다.

주어진 설정에서 그래프의 상태를 검사하려면 언제든지 `get_state(config)`를 호출할 수 있습니다. 이 메서드는 해당 `thread_id`에 저장된 가장 최근의 스냅샷을 반환합니다.

아래 코드에서는 `thread_id`가 `"1"`인 스레드의 상태를 조회하여 저장된 메시지를 확인합니다.

In [10]:
# thread_id "1"의 상태 조회
config = RunnableConfig(
    configurable={"thread_id": "1"},
)

# 그래프 상태 스냅샷 생성
snapshot = graph.get_state(config)

# 저장된 메시지 확인
snapshot.values["messages"]

[HumanMessage(content='내 이름은 `테디노트` 입니다. YouTube 채널을 운영하고 있어요. 만나서 반가워요', additional_kwargs={}, response_metadata={}, id='8cf273c7-3880-4484-9e06-ea7c64a195f3'),
 AIMessage(content=[{'text': '안녕하세요, 테디노트님! 만나서 반가워요! 🎬\n\nYouTube 채널을 운영하고 계시는군요. 어떤 주제나 콘텐츠로 채널을 운영하고 계신가요? 궁금합니다! \n\n제가 도움을 드릴 수 있는 부분이 있다면 언제든지 말씀해 주세요. 채널 운영, 콘텐츠 아이디어, 기술적인 부분 등 무엇이든 함께 이야기 나눌 수 있습니다. 😊', 'type': 'text', 'index': 0}], additional_kwargs={}, response_metadata={'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic', 'stop_reason': 'end_turn', 'stop_sequence': None}, id='lc_run--019c872b-19bc-70c1-bb3f-5da86b950d63', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 669, 'output_tokens': 187, 'total_tokens': 856, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}),
 HumanMessage(content='내 이름이 뭐라고 했지?', additional_kwargs={}, response_metadata={}, id='2867a2cf-1064-440b-b71b-0d1b475d6c43'),
 AIMessage(content=[{'text': '테디노트라고 하셨습니다! 😊\n\nYouTube 채널을 운영하고 계시

### 설정 정보 확인 (config)

`snapshot.config`를 통해 현재 스냅샷의 설정 정보를 확인할 수 있습니다. 여기에는 `thread_id`, `checkpoint_ns`, `checkpoint_id` 등의 정보가 포함되어 있습니다. `checkpoint_id`는 각 체크포인트를 고유하게 식별하는 UUID이며, 특정 시점의 상태로 롤백할 때 사용할 수 있습니다.

아래 코드에서는 스냅샷의 config 정보를 출력합니다.

In [11]:
# 설정된 config 정보
snapshot.config

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f110321-a962-61b2-8004-f35561b9a327'}}

### 저장된 상태 값 확인 (values)

`snapshot.values`를 통해 현재 체크포인트에 저장된 전체 상태 값을 확인할 수 있습니다. 여기에는 `messages` 키 아래에 지금까지의 대화 내용이 저장되어 있습니다. 각 메시지에는 발화자(human/ai), 내용(content), 고유 ID 등의 정보가 포함됩니다.

아래 코드에서는 저장된 상태 값을 출력합니다.

In [12]:
# 저장된 값(values)
snapshot.values

{'messages': [HumanMessage(content='내 이름은 `테디노트` 입니다. YouTube 채널을 운영하고 있어요. 만나서 반가워요', additional_kwargs={}, response_metadata={}, id='8cf273c7-3880-4484-9e06-ea7c64a195f3'),
  AIMessage(content=[{'text': '안녕하세요, 테디노트님! 만나서 반가워요! 🎬\n\nYouTube 채널을 운영하고 계시는군요. 어떤 주제나 콘텐츠로 채널을 운영하고 계신가요? 궁금합니다! \n\n제가 도움을 드릴 수 있는 부분이 있다면 언제든지 말씀해 주세요. 채널 운영, 콘텐츠 아이디어, 기술적인 부분 등 무엇이든 함께 이야기 나눌 수 있습니다. 😊', 'type': 'text', 'index': 0}], additional_kwargs={}, response_metadata={'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic', 'stop_reason': 'end_turn', 'stop_sequence': None}, id='lc_run--019c872b-19bc-70c1-bb3f-5da86b950d63', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 669, 'output_tokens': 187, 'total_tokens': 856, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}),
  HumanMessage(content='내 이름이 뭐라고 했지?', additional_kwargs={}, response_metadata={}, id='2867a2cf-1064-440b-b71b-0d1b475d6c43'),
  AIMessage(content=[{'text': '테디노트라고 하셨습니다! 😊\n\nYou

### 다음 노드 확인 (next)

`snapshot.next`를 통해 현재 시점에서 앞으로 방문할 다음 노드를 확인할 수 있습니다. 그래프가 `__END__`에 도달하여 실행이 완료된 경우, 다음 노드는 빈 값(`()`)이 출력됩니다. 반대로, 그래프가 아직 실행 중이거나 중단된 상태라면 다음에 실행될 노드의 이름이 표시됩니다.

아래 코드에서는 다음에 방문할 노드를 확인합니다.

In [13]:
# 다음 노드
snapshot.next

()

### 메타데이터 확인

`snapshot.metadata`에는 그래프 실행에 대한 상세 정보가 저장됩니다. 여기에는 실행 소스(source), 스텝 번호(step), 부모 체크포인트 정보(parents) 등이 포함됩니다. 이를 통해 그래프의 실행 흐름을 추적하고 디버깅할 수 있습니다.

아래 코드에서는 스냅샷의 메타데이터를 확인합니다.

In [14]:
# 스냅샷의 메타데이터 확인
snapshot.metadata

{'source': 'loop', 'step': 4, 'parents': {}}

### 메타데이터 시각화

스냅샷의 메타데이터는 중첩된 구조로 되어 있어 직접 확인하기 어려울 수 있습니다. `langchain_teddynote.messages` 모듈의 `display_message_tree()` 함수를 사용하면 트리 형태로 보기 쉽게 출력할 수 있습니다.

아래 코드에서는 스냅샷의 메타데이터를 트리 형태로 시각화합니다.

In [15]:
from langchain_teddynote.messages import display_message_tree

# 메타데이터를 트리 형태로 출력
display_message_tree(snapshot.metadata)

    source: "loop"
    step: 4
    parents: {}
